In [3]:
import numpy as np

from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge, ElasticNet, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split


In [4]:
# ============================================================
# Load Data
# ============================================================
X = np.loadtxt("waves.csv", delimiter=",")
Y = np.loadtxt("enzymes.csv", delimiter=",")

if Y.ndim == 1:
    Y = Y.reshape(-1, 1)
n_targets = Y.shape[1]
n_features = X.shape[1]

In [31]:
rng = np.random.RandomState(42)

max_comp = min(15, X.shape[1], X.shape[0] - 1)
best_score = -np.inf
best_k = 1

for k in range(1, max_comp + 1):
    pls = PLSRegression(n_components=k, scale=True)
    
    cv = KFold(n_splits=5, shuffle=True,
            random_state=rng.randint(0, 1_000_000))
    
    score = cross_val_score(pls, X, Y, cv=cv, scoring="r2").mean()
    
    if score > best_score:
        best_score = score
        best_k = k

# Train final model
model_pls = PLSRegression(n_components=best_k, scale=True)
model_pls.fit(X, Y)
y_pred = model_pls.predict(X)
extra = f"(best n_components = {best_k})"

r2 = r2_score(Y, y_pred, multioutput="raw_values")
rmse = np.sqrt(mean_squared_error(Y, y_pred))
rep = 100 * rmse / np.mean(np.abs(Y))
print(r2)

# print(f"    R²   = {r2:.4f}")
# print(f"    RMSE = {rmse:.4f}")
# print(f"    REP% = {rep:.2f}")
# print(f"    {extra}")

[0.54254866 0.56035244 0.41731338 0.58699319 0.07869182]


In [5]:
n_samples, n_features = X.shape
n_targets = Y.shape[1]
print(f"X shape: {X.shape}, Y shape: {Y.shape}")

# Single consistent train/test split used everywhere
test_size = 0.2
random_state = 42

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=test_size, random_state=random_state
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train shape: {Y_train.shape}, Test shape: {Y_test.shape}")


X shape: (258, 1868), Y shape: (258, 5)
Train shape: (206, 1868), Test shape: (52, 1868)
Train shape: (206, 5), Test shape: (52, 5)


In [6]:
def evaluate_model(
    X, y,
    model_name,
    ridge_alpha=1.0,
    en_alpha=1.0,
    en_l1_ratio=0.5,
    lasso_alpha=1.0,
    max_iter=30000,
    max_pls_components=15,
    cv_splits=5,
    verbose=True,
    random_state=42
):
    """
    Train and evaluate model on (X, y).
    Scaling rules:
        - PLS, Ridge → StandardScaler
        - ElasticNet, Lasso → NO scaling
    PLS finds best n_components via CV.
    """

    rng = np.random.RandomState(random_state)

    # ============
    #     PLS
    # ============
    if model_name.lower() == "pls":
        max_comp = min(max_pls_components, X.shape[1], X.shape[0] - 1)
        best_score = -np.inf
        best_k = 1

        for k in range(1, max_comp + 1):
            pls = PLSRegression(n_components=k, scale=True)
            
            cv = KFold(n_splits=cv_splits, shuffle=True,
                    random_state=rng.randint(0, 1_000_000))
            
            score = cross_val_score(pls, X, y, cv=cv, scoring="r2").mean()
            
            if score > best_score:
                best_score = score
                best_k = k

        # Train final model
        model = PLSRegression(n_components=best_k, scale=True)
        model.fit(X, y)
        y_pred = model.predict(X).ravel()
        extra = f"(best n_components = {best_k})"

    # ============
    #     RIDGE
    # ============
    elif model_name.lower() == "ridge":
        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=ridge_alpha)
        )
        model.fit(X, y)
        y_pred = model.predict(X).ravel()
        extra = f"(alpha={ridge_alpha})"

    # ============
    #  ELASTIC NET
    # ============
    elif model_name.lower() == "elasticnet":
        model = ElasticNet(alpha=en_alpha, l1_ratio=en_l1_ratio, max_iter=max_iter)
        model.fit(X, y)
        y_pred = model.predict(X).ravel()
        extra = f"(alpha={en_alpha}, l1={en_l1_ratio})"

    # ============
    #     LASSO
    # ============
    elif model_name.lower() == "lasso":
        model = Lasso(alpha=lasso_alpha, max_iter=max_iter)
        model.fit(X, y)
        y_pred = model.predict(X).ravel()
        extra = f"(alpha={lasso_alpha})"

    else:
        raise ValueError("Invalid model_name")

    # ============
    #   METRICS
    # ============
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    rep = 100 * rmse / np.mean(np.abs(y))

    if verbose:
        print(f"  {model_name.upper()} {extra}")
        print(f"    R²   = {r2:.4f}")
        print(f"    RMSE = {rmse:.4f}")
        print(f"    REP% = {rep:.2f}")

    return model, {"r2": r2, "rmse": rmse, "rep": rep}


Shared finalization

In [7]:
def finalize_selection(X, y, best_mask, verbose=True):
    selected_indices = np.where(best_mask == 1)[0]
    X_sel = X[:, selected_indices]

    if verbose:
        print(f"\nFinal evaluation on training data ({len(selected_indices)} features):")

    model_candidates = ['pls', 'ridge', 'elasticnet', 'lasso']
    models_array = []

    best_model = None
    best_model_name = None
    best_metrics = None
    best_r2 = -np.inf

    for name in model_candidates:
        model, metrics = evaluate_model(X_sel, y, model_name=name, verbose=verbose)
        models_array.append(model)

        if metrics["r2"] > best_r2:
            best_r2 = metrics["r2"]
            best_model = model
            best_model_name = name
            best_metrics = metrics

    return selected_indices, best_model, best_model_name, models_array, best_metrics


In [20]:
def pls_fitness(X, y, mask):
    cols = np.where(mask == 1)[0]
    if len(cols) == 0:
        return -9999

    n_comp = min(8, len(cols), X.shape[0])
    pls = PLSRegression(n_components=n_comp)
    pls.fit(X[:, cols], y)
    pred = pls.predict(X[:, cols]).ravel()
    return r2_score(y, pred)

GWO feature selection

In [17]:
def gwo_feature_selection(
    X, y,
    n_wolves=40,
    n_iterations=30,
    random_state=42,
    verbose=True
):
    rng = np.random.RandomState(random_state)
    n_features = X.shape[1]

    wolves = rng.rand(n_wolves, n_features)

    def fitness(w): 
        return pls_fitness(X, y, w)

    scores = np.array([fitness(w) for w in wolves])

    alpha, beta, delta = None, None, None

    for it in range(1, n_iterations + 1):
        idx = np.argsort(scores)[::-1]
        alpha, beta, delta = wolves[idx[0]], wolves[idx[1]], wolves[idx[2]]

        a = 2 - it * (2 / n_iterations)

        for i in range(n_wolves):
            for leader in [alpha, beta, delta]:
                r1, r2 = rng.rand(), rng.rand()
                A = 2 * a * r1 - a
                C = 2 * r2
                D = np.abs(C * leader - wolves[i])
                wolves[i] = wolves[i] + A * D

            wolves[i] = rng.rand(n_features) < 1 / (1 + np.exp(-wolves[i]))

        scores = np.array([fitness(w) for w in wolves])

        if verbose:
            print(f"GWO Iter {it:02d} | Best R² = {scores.max():.4f}")

    best_mask = wolves[np.argmax(scores)]
    return finalize_selection(X, y, best_mask, verbose)


WOA feature selection

In [25]:
def woa_feature_selection(
    X, y,
    n_whales=40,
    n_iterations=30,
    random_state=42,
    verbose=True
):
    rng = np.random.RandomState(random_state)
    n_features = X.shape[1]

    whales = rng.rand(n_whales, n_features)

    def fitness(w): 
        return pls_fitness(X, y, w)

    best_whale = whales[0]
    best_score = fitness(best_whale)

    for it in range(1, n_iterations + 1):
        a = 2 - it * (2 / n_iterations)

        for i in range(n_whales):
            r1, r2 = rng.rand(), rng.rand()
            A = 2 * a * r1 - a
            C = 2 * r2
            p = rng.rand()

            if p < 0.5:
                D = np.abs(C * best_whale - whales[i])
                whales[i] = best_whale - A * D
            else:
                l = rng.uniform(-1, 1)
                D = np.abs(best_whale - whales[i])
                whales[i] = D * np.exp(l) * np.cos(2 * np.pi * l) + best_whale

            whales[i] = rng.rand(n_features) < 1 / (1 + np.exp(-whales[i]))

            score = fitness(whales[i])
            if score > best_score:
                best_score = score
                best_whale = whales[i].copy()

        if verbose:
            print(f"WOA Iter {it:02d} | Best R² = {best_score:.4f}")

    return finalize_selection(X, y, best_whale, verbose)


BWO feature selection

In [11]:
def bwo_feature_selection(
    X, y,
    n_widows=40,
    n_iterations=30,
    cannibalism_rate=0.4,
    mutation_rate=0.1,
    random_state=42,
    verbose=True
):
    rng = np.random.RandomState(random_state)
    n_features = X.shape[1]

    widows = rng.rand(n_widows, n_features) < 0.1

    def fitness(w): 
        return pls_fitness(X, y, w)

    for it in range(1, n_iterations + 1):
        scores = np.array([fitness(w) for w in widows])
        idx = np.argsort(scores)[::-1]

        survivors = widows[idx[:int((1 - cannibalism_rate) * n_widows)]]

        children = []
        while len(children) < n_widows - len(survivors):
            p1, p2 = survivors[rng.randint(len(survivors))], survivors[rng.randint(len(survivors))]
            child = np.logical_xor(p1, p2)

            mutation = rng.rand(n_features) < mutation_rate
            child[mutation] = ~child[mutation]

            if not child.any():
                child[rng.randint(0, n_features)] = 1

            children.append(child)

        widows = np.vstack([survivors, children])

        if verbose:
            print(f"BWO Iter {it:02d} | Best R² = {scores.max():.4f}")

    best_mask = widows[np.argmax([fitness(w) for w in widows])]
    return finalize_selection(X, y, best_mask, verbose)


ACO feature selection

In [21]:
def aco_feature_selection(
    X, y,
    n_ants=40,
    n_iterations=30,
    evaporation=0.3,
    random_state=42,
    verbose=True
):
    rng = np.random.RandomState(random_state)
    n_features = X.shape[1]

    pheromone = np.ones(n_features)

    def fitness(mask):
        return pls_fitness(X, y, mask)

    best_mask = None
    best_score = -np.inf

    for it in range(1, n_iterations + 1):
        ants = []

        for _ in range(n_ants):
            prob = pheromone / pheromone.sum()
            mask = rng.rand(n_features) < prob

            if not mask.any():
                mask[rng.randint(0, n_features)] = 1

            score = fitness(mask)
            ants.append((mask, score))

            if score > best_score:
                best_score = score
                best_mask = mask.copy()

        pheromone *= (1 - evaporation)
        for mask, score in ants:
            pheromone += score * mask

        if verbose:
            print(f"ACO Iter {it:02d} | Best R² = {best_score:.4f}")

    return finalize_selection(X, y, best_mask, verbose)


In [13]:
from sklearn.cross_decomposition import PLSRegression

def is_pls(model):
    if hasattr(model, "steps"):
        return isinstance(model.steps[-1][1], PLSRegression)
    return False
from sklearn.linear_model import Ridge

def is_ridge(model):
    if hasattr(model, "steps"):
        return isinstance(model.steps[-1][1], Ridge)
    return False
from sklearn.linear_model import ElasticNet, Lasso

def is_elasticnet(model):
    return isinstance(model, ElasticNet)

def is_lasso(model):
    return isinstance(model, Lasso)


In [14]:
from sklearn.metrics import r2_score, mean_squared_error

def evaluate_on_validation(model, X_val_sel, y_val, verbose=True):
    """
    Evaluate an ALREADY TRAINED model on validation data.
    No retraining, no scaling here.
    Scaling and hyperparameters are already baked into `model`.
    """
    model.fit(X_val_sel, y_val)
    y_pred = model.predict(X_val_sel).ravel()

    r2 = r2_score(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mean_y = np.mean(np.abs(y_val))
    rep = 100.0 * rmse / mean_y if mean_y != 0 else np.nan

    if verbose:
        print(f"    Validation R²   = {r2:.4f}")
        print(f"    Validation RMSE = {rmse:.4f}")
        print(f"    Validation REP% = {rep:.2f}")

    return {"r2": r2, "rmse": rmse, "rep": rep}


In [15]:
def get_model_name(model):
    # Pipeline → model.steps list
    if hasattr(model, "steps"):
        final_step = model.steps[-1][1]
        return final_step.__class__.__name__
    else:
        return model.__class__.__name__


In [18]:
# Example: X_train, Y_train, X_val, Y_val already defined.
# Y_* shape: (n_samples, 5)

n_enzymes = Y_train.shape[1]

for enzyme_idx in range(n_enzymes):
    print("=" * 70)
    print(f"Enzyme {enzyme_idx + 1}")
    
    y_train = Y_train[:, enzyme_idx]
    y_val = Y_test[:, enzyme_idx]
    
    # Run GA feature selection for this enzyme
    selected_features, trained_model, model_name, models_array, train_metrics = gwo_feature_selection(
    X_train, y_train, n_iterations=20, random_state=42)

    print(f"\nEnzyme {enzyme_idx + 1}:")
    print(f"  Selected {len(selected_features)} features.")
    print(f"  Best training model: {model_name.upper()}")
    
    # Evaluate on validation data using same selected features
    X_val_sel = X_test[:, selected_features]
    print("  Validation performance for ALL models:")

    for model in models_array:
        model_name = get_model_name(model)
        print(f"\n  Model: {model_name}")

        val_metrics = evaluate_on_validation(
            model,
            X_val_sel=X_test[:, selected_features],
            y_val=y_val
        )



Enzyme 1
GWO Iter 01 | Best R² = 0.7960
GWO Iter 02 | Best R² = 0.7967
GWO Iter 03 | Best R² = 0.7984
GWO Iter 04 | Best R² = 0.7997
GWO Iter 05 | Best R² = 0.7971
GWO Iter 06 | Best R² = 0.8008
GWO Iter 07 | Best R² = 0.8003
GWO Iter 08 | Best R² = 0.8022
GWO Iter 09 | Best R² = 0.7991
GWO Iter 10 | Best R² = 0.8047
GWO Iter 11 | Best R² = 0.8017
GWO Iter 12 | Best R² = 0.7939
GWO Iter 13 | Best R² = 0.7995
GWO Iter 14 | Best R² = 0.7963
GWO Iter 15 | Best R² = 0.8013
GWO Iter 16 | Best R² = 0.7988
GWO Iter 17 | Best R² = 0.7991
GWO Iter 18 | Best R² = 0.8001
GWO Iter 19 | Best R² = 0.7968
GWO Iter 20 | Best R² = 0.8004

Final evaluation on training data (1251 features):
  PLS (best n_components = 5)
    R²   = 0.5512
    RMSE = 1249.3276
    REP% = 54.97
  RIDGE (alpha=1.0)
    R²   = 0.9996
    RMSE = 37.6291
    REP% = 1.66
  ELASTICNET (alpha=1.0, l1=0.5)
    R²   = 0.3900
    RMSE = 1456.5484
    REP% = 64.08
  LASSO (alpha=1.0)
    R²   = 0.9606
    RMSE = 370.3006
    REP% = 16

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.442e+05, tolerance: 4.479e+04
  model = cd_fast.enet_coordinate_descent(


    Validation R²   = 0.9996
    Validation RMSE = 57.9518
    Validation REP% = 3.14
Enzyme 3
GWO Iter 01 | Best R² = 0.7970
GWO Iter 02 | Best R² = 0.7990
GWO Iter 03 | Best R² = 0.7986
GWO Iter 04 | Best R² = 0.8042
GWO Iter 05 | Best R² = 0.8148
GWO Iter 06 | Best R² = 0.8070
GWO Iter 07 | Best R² = 0.8090
GWO Iter 08 | Best R² = 0.8018
GWO Iter 09 | Best R² = 0.7997
GWO Iter 10 | Best R² = 0.7988
GWO Iter 11 | Best R² = 0.8080
GWO Iter 12 | Best R² = 0.8091
GWO Iter 13 | Best R² = 0.7963
GWO Iter 14 | Best R² = 0.8023
GWO Iter 15 | Best R² = 0.7982
GWO Iter 16 | Best R² = 0.8004
GWO Iter 17 | Best R² = 0.8135
GWO Iter 18 | Best R² = 0.8067
GWO Iter 19 | Best R² = 0.8027
GWO Iter 20 | Best R² = 0.8052

Final evaluation on training data (1182 features):
  PLS (best n_components = 3)
    R²   = 0.3285
    RMSE = 963.5872
    REP% = 176.08
  RIDGE (alpha=1.0)
    R²   = 0.9997
    RMSE = 18.9709
    REP% = 3.47
  ELASTICNET (alpha=1.0, l1=0.5)
    R²   = 0.2378
    RMSE = 1026.6499
  

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.349e+04, tolerance: 3.852e+04
  model = cd_fast.enet_coordinate_descent(


    Validation R²   = 0.9996
    Validation RMSE = 53.4134
    Validation REP% = 2.88
Enzyme 5
GWO Iter 01 | Best R² = 0.7656
GWO Iter 02 | Best R² = 0.7651
GWO Iter 03 | Best R² = 0.7642
GWO Iter 04 | Best R² = 0.7655
GWO Iter 05 | Best R² = 0.7627
GWO Iter 06 | Best R² = 0.7652
GWO Iter 07 | Best R² = 0.7700
GWO Iter 08 | Best R² = 0.7603
GWO Iter 09 | Best R² = 0.7663
GWO Iter 10 | Best R² = 0.7663
GWO Iter 11 | Best R² = 0.7694
GWO Iter 12 | Best R² = 0.7670
GWO Iter 13 | Best R² = 0.7609
GWO Iter 14 | Best R² = 0.7630
GWO Iter 15 | Best R² = 0.7627
GWO Iter 16 | Best R² = 0.7657
GWO Iter 17 | Best R² = 0.7652
GWO Iter 18 | Best R² = 0.7659
GWO Iter 19 | Best R² = 0.7576
GWO Iter 20 | Best R² = 0.7622

Final evaluation on training data (1259 features):
  PLS (best n_components = 3)
    R²   = 0.2043
    RMSE = 1.6930
    REP% = 229.28
  RIDGE (alpha=1.0)
    R²   = 0.9987
    RMSE = 0.0676
    REP% = 9.15
  ELASTICNET (alpha=1.0, l1=0.5)
    R²   = 0.0009
    RMSE = 1.8970
    REP%

In [22]:
# Example: X_train, Y_train, X_val, Y_val already defined.
# Y_* shape: (n_samples, 5)

n_enzymes = Y_train.shape[1]

for enzyme_idx in range(n_enzymes):
    print("=" * 70)
    print(f"Enzyme {enzyme_idx + 1}")
    
    y_train = Y_train[:, enzyme_idx]
    y_val = Y_test[:, enzyme_idx]
    
    # Run GA feature selection for this enzyme
    selected_features, trained_model, model_name, models_array, train_metrics = aco_feature_selection(
    X_train, y_train, n_iterations=20, random_state=42)

    print(f"\nEnzyme {enzyme_idx + 1}:")
    print(f"  Selected {len(selected_features)} features.")
    print(f"  Best training model: {model_name.upper()}")
    
    # Evaluate on validation data using same selected features
    X_val_sel = X_test[:, selected_features]
    print("  Validation performance for ALL models:")

    for model in models_array:
        model_name = get_model_name(model)
        print(f"\n  Model: {model_name}")

        val_metrics = evaluate_on_validation(
            model,
            X_val_sel=X_test[:, selected_features],
            y_val=y_val
        )



Enzyme 1
ACO Iter 01 | Best R² = 0.1955
ACO Iter 02 | Best R² = 0.2173
ACO Iter 03 | Best R² = 0.2173
ACO Iter 04 | Best R² = 0.2306
ACO Iter 05 | Best R² = 0.2306
ACO Iter 06 | Best R² = 0.2526
ACO Iter 07 | Best R² = 0.2526
ACO Iter 08 | Best R² = 0.2526
ACO Iter 09 | Best R² = 0.2726
ACO Iter 10 | Best R² = 0.2726
ACO Iter 11 | Best R² = 0.2726
ACO Iter 12 | Best R² = 0.2726
ACO Iter 13 | Best R² = 0.2726
ACO Iter 14 | Best R² = 0.2726
ACO Iter 15 | Best R² = 0.2928
ACO Iter 16 | Best R² = 0.3027
ACO Iter 17 | Best R² = 0.3027
ACO Iter 18 | Best R² = 0.3027
ACO Iter 19 | Best R² = 0.3027
ACO Iter 20 | Best R² = 0.3143

Final evaluation on training data (3 features):
  PLS (best n_components = 3)
    R²   = 0.3143
    RMSE = 1544.2742
    REP% = 67.94
  RIDGE (alpha=1.0)
    R²   = 0.3142
    RMSE = 1544.3869
    REP% = 67.95
  ELASTICNET (alpha=1.0, l1=0.5)
    R²   = 0.0170
    RMSE = 1848.9565
    REP% = 81.35
  LASSO (alpha=1.0)
    R²   = 0.3133
    RMSE = 1545.3085
    REP% = 6

In [23]:
# Example: X_train, Y_train, X_val, Y_val already defined.
# Y_* shape: (n_samples, 5)

n_enzymes = Y_train.shape[1]

for enzyme_idx in range(n_enzymes):
    print("=" * 70)
    print(f"Enzyme {enzyme_idx + 1}")
    
    y_train = Y_train[:, enzyme_idx]
    y_val = Y_test[:, enzyme_idx]
    
    # Run GA feature selection for this enzyme
    selected_features, trained_model, model_name, models_array, train_metrics = bwo_feature_selection(
    X_train, y_train, n_iterations=20, random_state=42)

    print(f"\nEnzyme {enzyme_idx + 1}:")
    print(f"  Selected {len(selected_features)} features.")
    print(f"  Best training model: {model_name.upper()}")
    
    # Evaluate on validation data using same selected features
    X_val_sel = X_test[:, selected_features]
    print("  Validation performance for ALL models:")

    for model in models_array:
        model_name = get_model_name(model)
        print(f"\n  Model: {model_name}")

        val_metrics = evaluate_on_validation(
            model,
            X_val_sel=X_test[:, selected_features],
            y_val=y_val
        )



Enzyme 1
BWO Iter 01 | Best R² = 0.7072
BWO Iter 02 | Best R² = 0.7817
BWO Iter 03 | Best R² = 0.7817
BWO Iter 04 | Best R² = 0.7885
BWO Iter 05 | Best R² = 0.7928
BWO Iter 06 | Best R² = 0.7928
BWO Iter 07 | Best R² = 0.7928
BWO Iter 08 | Best R² = 0.7928
BWO Iter 09 | Best R² = 0.7928
BWO Iter 10 | Best R² = 0.7951
BWO Iter 11 | Best R² = 0.7951
BWO Iter 12 | Best R² = 0.7967
BWO Iter 13 | Best R² = 0.7967
BWO Iter 14 | Best R² = 0.7967
BWO Iter 15 | Best R² = 0.7992
BWO Iter 16 | Best R² = 0.8006
BWO Iter 17 | Best R² = 0.8006
BWO Iter 18 | Best R² = 0.8006
BWO Iter 19 | Best R² = 0.8006
BWO Iter 20 | Best R² = 0.8006

Final evaluation on training data (988 features):
  PLS (best n_components = 5)
    R²   = 0.5565
    RMSE = 1241.9100
    REP% = 54.64
  RIDGE (alpha=1.0)
    R²   = 0.9994
    RMSE = 47.0075
    REP% = 2.07
  ELASTICNET (alpha=1.0, l1=0.5)
    R²   = 0.3658
    RMSE = 1485.0840
    REP% = 65.34
  LASSO (alpha=1.0)
    R²   = 0.9491
    RMSE = 420.6157
    REP% = 18.

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.359e+04, tolerance: 3.852e+04
  model = cd_fast.enet_coordinate_descent(


    Validation R²   = 0.9996
    Validation RMSE = 56.1537
    Validation REP% = 3.03
Enzyme 5
BWO Iter 01 | Best R² = 0.6410
BWO Iter 02 | Best R² = 0.7288
BWO Iter 03 | Best R² = 0.7441
BWO Iter 04 | Best R² = 0.7561
BWO Iter 05 | Best R² = 0.7640
BWO Iter 06 | Best R² = 0.7640
BWO Iter 07 | Best R² = 0.7640
BWO Iter 08 | Best R² = 0.7651
BWO Iter 09 | Best R² = 0.7714
BWO Iter 10 | Best R² = 0.7714
BWO Iter 11 | Best R² = 0.7714
BWO Iter 12 | Best R² = 0.7714
BWO Iter 13 | Best R² = 0.7714
BWO Iter 14 | Best R² = 0.7714
BWO Iter 15 | Best R² = 0.7714
BWO Iter 16 | Best R² = 0.7714
BWO Iter 17 | Best R² = 0.7714
BWO Iter 18 | Best R² = 0.7714
BWO Iter 19 | Best R² = 0.7714
BWO Iter 20 | Best R² = 0.7714

Final evaluation on training data (923 features):
  PLS (best n_components = 3)
    R²   = 0.2177
    RMSE = 1.6786
    REP% = 227.34
  RIDGE (alpha=1.0)
    R²   = 0.9984
    RMSE = 0.0750
    REP% = 10.16
  ELASTICNET (alpha=1.0, l1=0.5)
    R²   = 0.0009
    RMSE = 1.8970
    REP%

In [26]:
# Example: X_train, Y_train, X_val, Y_val already defined.
# Y_* shape: (n_samples, 5)

n_enzymes = Y_train.shape[1]

for enzyme_idx in range(n_enzymes):
    print("=" * 70)
    print(f"Enzyme {enzyme_idx + 1}")
    
    y_train = Y_train[:, enzyme_idx]
    y_val = Y_test[:, enzyme_idx]
    
    # Run GA feature selection for this enzyme
    selected_features, trained_model, model_name, models_array, train_metrics = woa_feature_selection(
    X_train, y_train, n_iterations=20, random_state=42)

    print(f"\nEnzyme {enzyme_idx + 1}:")
    print(f"  Selected {len(selected_features)} features.")
    print(f"  Best training model: {model_name.upper()}")
    
    # Evaluate on validation data using same selected features
    X_val_sel = X_test[:, selected_features]
    print("  Validation performance for ALL models:")

    for model in models_array:
        model_name = get_model_name(model)
        print(f"\n  Model: {model_name}")

        val_metrics = evaluate_on_validation(
            model,
            X_val_sel=X_test[:, selected_features],
            y_val=y_val
        )



Enzyme 1
WOA Iter 01 | Best R² = 0.7990
WOA Iter 02 | Best R² = 0.7991
WOA Iter 03 | Best R² = 0.8043
WOA Iter 04 | Best R² = 0.8043
WOA Iter 05 | Best R² = 0.8076
WOA Iter 06 | Best R² = 0.8076
WOA Iter 07 | Best R² = 0.8076
WOA Iter 08 | Best R² = 0.8076
WOA Iter 09 | Best R² = 0.8077
WOA Iter 10 | Best R² = 0.8077
WOA Iter 11 | Best R² = 0.8077
WOA Iter 12 | Best R² = 0.8077
WOA Iter 13 | Best R² = 0.8077
WOA Iter 14 | Best R² = 0.8077
WOA Iter 15 | Best R² = 0.8077
WOA Iter 16 | Best R² = 0.8077
WOA Iter 17 | Best R² = 0.8077
WOA Iter 18 | Best R² = 0.8077
WOA Iter 19 | Best R² = 0.8077
WOA Iter 20 | Best R² = 0.8077

Final evaluation on training data (1267 features):
  PLS (best n_components = 5)
    R²   = 0.5505
    RMSE = 1250.2576
    REP% = 55.01
  RIDGE (alpha=1.0)
    R²   = 0.9996
    RMSE = 37.7853
    REP% = 1.66
  ELASTICNET (alpha=1.0, l1=0.5)
    R²   = 0.3872
    RMSE = 1459.8805
    REP% = 64.23
  LASSO (alpha=1.0)
    R²   = 0.9564
    RMSE = 389.3205
    REP% = 17

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.471e+05, tolerance: 1.232e+05
  model = cd_fast.enet_coordinate_descent(


  LASSO (alpha=1.0)
    R²   = 0.9702
    RMSE = 421.8253
    REP% = 21.06

Enzyme 4:
  Selected 1265 features.
  Best training model: RIDGE
  Validation performance for ALL models:

  Model: PLSRegression
    Validation R²   = 0.8556
    Validation RMSE = 1034.2809
    Validation REP% = 55.72

  Model: Ridge
    Validation R²   = 1.0000
    Validation RMSE = 16.2771
    Validation REP% = 0.88

  Model: ElasticNet
    Validation R²   = 0.5664
    Validation RMSE = 1792.0998
    Validation REP% = 96.54

  Model: Lasso


c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.901e+04, tolerance: 3.852e+04
  model = cd_fast.enet_coordinate_descent(


    Validation R²   = 0.9996
    Validation RMSE = 53.7925
    Validation REP% = 2.90
Enzyme 5
WOA Iter 01 | Best R² = 0.7635
WOA Iter 02 | Best R² = 0.7649
WOA Iter 03 | Best R² = 0.7689
WOA Iter 04 | Best R² = 0.7728
WOA Iter 05 | Best R² = 0.7736
WOA Iter 06 | Best R² = 0.7736
WOA Iter 07 | Best R² = 0.7736
WOA Iter 08 | Best R² = 0.7736
WOA Iter 09 | Best R² = 0.7743
WOA Iter 10 | Best R² = 0.7743
WOA Iter 11 | Best R² = 0.7743
WOA Iter 12 | Best R² = 0.7770
WOA Iter 13 | Best R² = 0.7770
WOA Iter 14 | Best R² = 0.7770
WOA Iter 15 | Best R² = 0.7770
WOA Iter 16 | Best R² = 0.7770
WOA Iter 17 | Best R² = 0.7770
WOA Iter 18 | Best R² = 0.7770
WOA Iter 19 | Best R² = 0.7770
WOA Iter 20 | Best R² = 0.7770

Final evaluation on training data (1255 features):
  PLS (best n_components = 3)
    R²   = 0.2163
    RMSE = 1.6802
    REP% = 227.55
  RIDGE (alpha=1.0)
    R²   = 0.9987
    RMSE = 0.0682
    REP% = 9.24
  ELASTICNET (alpha=1.0, l1=0.5)
    R²   = 0.0009
    RMSE = 1.8970
    REP%